### Script to do Foward run.

What to do first:
* Create a production directory (if it doesn't exist already) and place "insert_file_name_here.ipynb", "wrf_settings.py", and "make_namelist.py".
* In that directory create an "em_fwd" directory.
* In em_adj, the following files are required:
    * GENPARM.TBL
    * LANDUSE.TBL
    * RRTMG_LW_DATA
    * RRTMG_SW_DATA
    * RRTM_DATA
    * VEGPARM.TBL
    * wrf.exe
    * plus.io_config
    * wrfinput_d01
    * wrfbdy_d01




In [ ]:
import numpy as np
import netCDF4
from netCDF4 import Dataset
from wrf import getvar
import os
import subprocess
from subprocess import PIPE
import sys
import glob
import importlib
import pandas as pd
import xarray as xr

sys.path.append(os.path.abspath('..'))

import wrf_settings
import make_namelist

In [2]:
#Select experiment to load settings for

EXP_NAME = 'WRF_Florence_test'

importlib.reload(wrf_settings)

settings = wrf_settings.get_settings(EXP_NAME)
WRF_DIR           = settings['WRF_dir']
BOX_SIZE          = settings['box_size']
ADJ_JC            = settings['adj_jc']
ADJ_IC            = settings['adj_ic']

RUN_HOURS         = settings['run_hours']
START_YEAR          = settings['start_year']
START_MONTH         = settings['start_month']
START_DAY           = settings['start_day']
START_HOUR          = settings['start_hour']
END_YEAR            = settings['end_year']
END_MONTH           = settings['end_month']
END_DAY             = settings['end_day']
END_HOUR            = settings['end_hour']
E_WE               = settings['e_we']
E_SN               = settings['e_sn']
DX                 = settings['dx']
DY                 = settings['dy']
TIME_STEP          = settings['time_step']
INTERVAL_SECONDS   = settings['interval_seconds']
INTERVAL_SECONDS_ADJ = settings['interval_seconds_adj']
DRESPONSE_VALUE = settings['dresponse_value']


print(settings)



{'WRF_dir': '/Users/ngordillo/florence/', 'adj_jc': 119, 'adj_ic': 181, 'box_size': 10, 'run_hours': '36', 'start_year': '2018', 'start_month': '09', 'start_day': '09', 'start_hour': '00', 'end_year': '2018', 'end_month': '09', 'end_day': '10', 'end_hour': '12', 'interval_seconds': '3600', 'interval_seconds_adj': '10800', 'time_step': '60', 'e_we': 350, 'e_sn': 240, 'dx': 18000, 'dy': 18000, 'dresponse_value': -150}


In [3]:
# Create experiment directory and subdirectories for namelists and WRF data (if not created already)

os.chdir(WRF_DIR)
command_mkdir = "mkdir exp_files"
output_mkdir=subprocess.run(command_mkdir,shell=True, stdout=PIPE)

os.chdir(WRF_DIR + "exp_files/")
command_mkdir = "mkdir " + EXP_NAME
output_mkdir=subprocess.run(command_mkdir,shell=True, stdout=PIPE)

os.chdir(WRF_DIR + "exp_files/" + EXP_NAME)      

command_mkdir = "mkdir namelists"
output_mkdir=subprocess.run(command_mkdir,shell=True, stdout=PIPE)

command_mkdir = "mkdir wrf_data"
output_mkdir=subprocess.run(command_mkdir,shell=True, stdout=PIPE)

mkdir: cannot create directory ‘exp_files’: File exists
mkdir: cannot create directory ‘WRF_Florence_test’: File exists
mkdir: cannot create directory ‘namelists’: File exists
mkdir: cannot create directory ‘wrf_data’: File exists


### Section 1: Forward Run

In [4]:
# make dirs for namelist, and outputsif they don't exist and wrfinputs if they don't exist

output_file = os.path.join(WRF_DIR, "exp_files/" + EXP_NAME + "/namelists/namelist.input.fwd." + EXP_NAME) 

proceed = True

#Comment out the following block if you want to overwrite existing namelist without warning.

if os.path.exists(output_file):
    response = input(f"Warning: A namelist already exists at {output_file}. Overwrite it? (y/n): ")
    
    if response.lower() not in ['y', 'yes']:
        print("Skipping namelist generation.")
        proceed = False 
        
####

if(proceed):

    make_namelist.generate_namelist(
        run_hours = RUN_HOURS,
        start_year = START_YEAR,
        start_month = START_MONTH,
        start_day = START_DAY,
        start_hour = START_HOUR,
        end_year = END_YEAR,
        end_month = END_MONTH,
        end_day = END_DAY,
        end_hour = END_HOUR,
        time_step = TIME_STEP,
        interval_seconds = INTERVAL_SECONDS,
        e_we = E_WE,
        e_sn = E_SN,
        dx = DX,
        dy = DY,    
        wrf_dir = WRF_DIR,
        exp_name = EXP_NAME,

        run_type="fwd"

    )


Success! 'namelist.input.fwd.WRF_Florence_test' has been generated.


In [ ]:

#Link FWD namelist to namelist.input

os.chdir(WRF_DIR + 'em_fwd/')
command_linktlm = "ln -sf " + WRF_DIR + "exp_files/" + EXP_NAME + "/namelists/namelist.input.fwd." + EXP_NAME + " namelist.input"
output_linktlm=subprocess.run(command_linktlm,shell=True, stdout=PIPE)


In [ ]:
#Run WRF for forward run
command_runwrf = "source ~/.bashrc && mpirun -np 40 ./wrf.exe"
output_runwrf = subprocess.run(command_runwrf, shell=True, executable='/bin/bash', stdout=subprocess.PIPE)

 starting wrf task            0  of           40
 starting wrf task            1  of           40
 starting wrf task            2  of           40
 starting wrf task            3  of           40
 starting wrf task            4  of           40
 starting wrf task            5  of           40
 starting wrf task            6  of           40
 starting wrf task            7  of           40
 starting wrf task            8  of           40
 starting wrf task           10  of           40
 starting wrf task           11  of           40
 starting wrf task           12  of           40
 starting wrf task           13  of           40
 starting wrf task           14  of           40
 starting wrf task           15  of           40
 starting wrf task           16  of           40
 starting wrf task           17  of           40
 starting wrf task           18  of           40
 starting wrf task           19  of           40
 starting wrf task           20  of           40
 starting wrf task  

### Section 2: Calculate R  

In [4]:
# Load WRF output and extract variables needed to calculate R. 


fwd_wrf_file = WRF_DIR + '/em_fwd/wrfout_d01_' + START_YEAR + '-' + START_MONTH + '-' + START_DAY + '_' + START_HOUR + ':00:00'    # Your filename
fwd_wrf_ds = Dataset(fwd_wrf_file, 'r')  # Dataset is the class behavior to open the file

itime = -1 # select model time
msfm=getvar(fwd_wrf_ds, "MAPFAC_M", timeidx=itime,meta=False)   # Map scale factor on mass grid
slp=getvar(fwd_wrf_ds, "slp", timeidx=itime,meta=False)   # Map scale factor on mass grid
mup=getvar(fwd_wrf_ds, "MU", timeidx=itime,meta=False)   # Map scale factor on mass grid
G_MU=getvar(fwd_wrf_ds, "G_MU", timeidx=itime,meta=False)   # Map scale factor on mass grid
u = getvar(fwd_wrf_ds, "U",timeidx=-1,meta=False)
ds = 1./fwd_wrf_ds.variables['RDX'][0]     # grid spacing SAME IN ZONAL AND MERIDIONAL DIRECTIONS


In [5]:
#Get dimensions and initialize arrays for vorticity and Coriolis parameter

num_levs=u.shape[0]
num_lats = msfm.shape[0]
num_lons = msfm.shape[1]

In [6]:
#Calculate response function, R_mu = average(mu')
R_mu=0

#
n = 0
jc = ADJ_JC
ic = ADJ_IC

for j in np.arange(jc-BOX_SIZE,jc+BOX_SIZE):
    for i in np.arange(ic-BOX_SIZE,ic+BOX_SIZE): 
        #if(G_MU[j,i]<=0):
            R_mu += -mup[j,i]
            n += 1
# print(n)            
R_mu = R_mu/n
print(n, R_mu)         

400 455.2194415449025


In [ ]:
# #Optionally, save the response function value to a text file for later use in the adjoint run
# with open(os.path.join(WRF_DIR, "exp_files/" + EXP_NAME, "response_function_value.txt"), "w") as f:
#     f.write(str(R_mu))  


NameError: name 'R_mu' is not defined